# Tahap 1: Persiapan, Validasi, dan Pra-pemrosesan Data Draft Pick
**Penelitian:** HOK Virtual Coach: Model Rekomendasi Draft Pick pada Game Honor of Kings Berbasis Association Rule Mining (ARM) dan Long Short-Term Memory (LSTM)  
**Oleh:** Adiyatma Admaja - NIM 082111633061 (S-1 Sistem Informasi, Universitas Airlangga)  
**Referensi Naskah:** BAB III Subbab 3.1 (Persiapan Data) & BAB IV (Hasil Pengumpulan, Validasi, dan Pra-pemrosesan Data)  
**Urutan Pipeline:** Tahap 1 dari 5 (Prasyarat utama bagi Notebook 2, 3, 4, dan 5)

## 1. Tujuan Notebook
Notebook ini bertujuan melakukan penggabungan data mentah lintas musim, pemeriksaan kualitas data melalui triangulasi sumber (makro, mikro, dan kronologis), pembersihan pertandingan anomali, serta transformasi representasi fitur (*encoding* dan *sequence padding*) untuk menghasilkan dataset terstandarisasi bagi pemodelan ARM, LSTM, dan MLP.

## 2. Spesifikasi Input
* **Dataset Draft Pick Mentah (3 Musim IKL):** `Dataset_Utama - Spring25_Draft_Pick.csv` (187 pertandingan), `Fall25_Draft_Pick.csv` (177 pertandingan), dan `Spring26_Draft_Pick.csv` (205 pertandingan) dengan total **569 pertandingan (10.242 baris aksi draft)**.
* **Dataset Referensi Hero:** `(CSV) Dataset_Utama - Hero_List.csv` berisi **113 entitas hero unik** beserta atribut `Hero_ID`, `Hero_Name`, `Hero_Tipe`, dan `Default_Role`.
* **Dataset Referensi Liquipedia:** Rekapitulasi statistik sekunder (`Liquipedia_{Musim}.csv` dan `Liquipedia_Ringkasan_{Musim}.csv`) untuk verifikasi triangulasi makro.

## 3. Alur Proses Utama
1. **Triangulasi Makro, Mikro, & Kronologis (Subbab 3.1.2.a–c):** Mencocokkan statistik agregat per musim dengan Liquipedia serta menarik sampel acak 5% (`seed=42`) sejumlah **28 pertandingan (504 aksi draft)** untuk verifikasi kesesuaian entitas, *seating order role* (`Clash_Lane -> Jungler -> Mid_Lane -> Farm_Lane -> Roamer`), dan ketepatan urutan waktu terhadap VOD.
2. **Eksklusi Anomali & Pembersihan Data (Subbab 3.1.2.d):** Mengeksklusi **11 pertandingan tidak standar** (10 pertandingan karena *missing value* akibat ronde *Ultimate Battle / No Ban Penalty* dan 1 pertandingan karena duplikasi *hero / mirror pick*), sehingga menghasilkan **558 pertandingan bersih (10.044 baris aksi draft)** yang terdiri atas 184 pertandingan Spring 2025, 174 pertandingan Fall 2025, dan 200 pertandingan Spring 2026.
3. **Integrasi Data & Label Encoding (Subbab 3.1.3.a–b):** Menyatukan atribut referensi *hero*, menentukan label kemenangan per tim (`Is_Win`), dan memetakan `Hero_ID` ke indeks `Encoded_Hero` (`1` s.d. `113`, dengan indeks `0` untuk *padding*; `TOTAL_HEROES_VOCAB = 114`).
4. **Transformasi Matriks Spesifik Model & Agregasi Statistik (Subbab 3.1.3.b–c):**
   * *Data Transaksional ARM:* Memisahkan aksi pemilihan (`5.580 Picks`) dan pelarangan (`4.464 Bans`), serta menghitung statistik agregat kemunculan dan *Win Rate* per *hero*.
   * *Matriks Multi-Hot Encoding MLP:* Membentuk vektor berdimensi 114 per pertandingan (`1` = *pick* Tim Biru, `-1` = *pick* Tim Merah, `0` = tidak dipilih).
   * *Matriks Prefix Sequence LSTM:* Membentuk 17 urutan bertahap per pertandingan (total **9.486 sekuens**) dengan *pre-padding* bernilai `0` sepanjang 17 langkah beserta penanda grup pertandingan (`groups_lstm`).

## 4. Artefak Output yang Dihasilkan
* **Laporan Validasi:** `Hasil_Triangulasi_{Musim}.csv`, `Ringkasan_Makro_{Musim}.csv`, `Validasi_Mikro_Sampling_5_Persen.csv`, dan `Validasi_Kronologis_Sampling_5_Persen.csv`.
* **Dataset Bersih & Referensi:** `draft_combined_cleaned.csv`, `arm_picks_cleaned.csv`, `arm_bans_cleaned.csv`, `hero_data.csv`, dan `hero_stats_{Musim}.csv`.
* **Input Model & Parameter Global:** `X_mlp.npy` `(558, 114)`, `y_mlp.npy` `(558,)`, `X_lstm.npy` `(9486, 17)`, `y_lstm.npy` `(9486,)`, `groups_lstm.npy` `(9486,)`, serta `global_parameters.pkl`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.sequence import pad_sequences
import warnings
import random
import json
import os
from IPython.display import display
warnings.filterwarnings('ignore')

In [ ]:
# ==============================================================================
# 1. PENGUMPULAN & PENGGABUNGAN DATA AWAL (BAB 3.1.1)
# ==============================================================================
# Mendefinisikan path file dataset masing-masing musim
file_draft_spring_25 = "../Data/Dataset_Utama - Spring25_Draft_Pick.csv"
file_draft_fall_25   = "../Data/Dataset_Utama - Fall25_Draft_Pick.csv"
file_draft_spring_26 = "../Data/Dataset_Utama - Spring26_Draft_Pick.csv"
file_hero            = "../Data/(CSV) Dataset_Utama - Hero_List.csv"

# Membaca masing-masing dataset musim
df_spring_25 = pd.read_csv(file_draft_spring_25)
df_fall_25   = pd.read_csv(file_draft_fall_25)
df_spring_26 = pd.read_csv(file_draft_spring_26)
# Membaca dataset daftar hero
df_hero = pd.read_csv(file_hero)

# Menggabungkan ketiga dataset musim menjadi satu dataset utama
df_draft = pd.concat([df_spring_25, df_fall_25, df_spring_26], ignore_index=True)

# Menyeragamkan nomor urut baris 'NO' dan 'Match_Sequence' awal secara kontinu
df_draft['NO'] = range(1, len(df_draft) + 1)
df_draft['Match_Sequence'] = pd.factorize(df_draft['Match_ID'])[0] + 1

# Validasi sederhana dimensi data
print("="*60)
print("DATASET AWAL BERHASIL DIMUAT & DIGABUNGKAN")
print("="*60)
print(f"Total Pertandingan Spring 25    : {df_spring_25['Match_ID'].nunique()}")
print(f"Total Pertandingan Fall 25      : {df_fall_25['Match_ID'].nunique()}")
print(f"Total Pertandingan Spring 26    : {df_spring_26['Match_ID'].nunique()}")
print(f"Total Pertandingan Gabungan     : {df_draft['Match_ID'].nunique()}")
print(f"Total Baris Data Draft Gabungan : {df_draft.shape[0]}")
print(f"Total Hero Unik Tersedia        : {df_hero['Hero_ID'].nunique()}")

DATASET AWAL BERHASIL DIMUAT & DIGABUNGKAN
Total Pertandingan Spring 25    : 187
Total Pertandingan Fall 25      : 177
Total Pertandingan Spring 26    : 205
Total Pertandingan Gabungan     : 569
Total Baris Data Draft Gabungan : 10242
Total Hero Unik Tersedia        : 113


In [ ]:
print("="*60)
print("DATASET DRAFT PICK TIAP MUSIM")
print("="*60)

print(f"Dataset Draft pick IKL Spring 25")
print("="*60)
display(df_spring_25)

print("="*60)
print(f"Dataset Draft pick IKL Fall 25")
print("="*60)
display(df_fall_25)

print("="*60)
print(f"Dataset Draft pick IKL Spring 26")
print("="*60)
display(df_spring_26)

print("="*60)
print(f"Dataset Draft pick Gabungan")
print("="*60)
display(df_draft)

print("="*60)
print("DATASET HERO LIST")
print("="*60)

display(df_hero.head())

DATASET DRAFT PICK TIAP MUSIM
Dataset Draft pick IKL Spring 25


,NO,Tanggal_Match,Match_ID,Match_Sequence,Tournament_Season,Team_Name,Team_Side,Draft_Order,Is_Ban,Hero_ID,Hero_Name,Hero_Tipe,Hero_Role,Match_Outcome
0,1,4/11/2025,Match_20250411_1,1,Spring_2025,RRQ,Blue,1,True,564.0,Mayene,Fighter,Clash_Lane,Red_Win
1,2,4/11/2025,Match_20250411_1,1,Spring_2025,NMSS,Red,2,True,159.0,Dolia,Support,Roamer,Red_Win
2,3,4/11/2025,Match_20250411_1,1,Spring_2025,RRQ,Blue,3,True,108.0,Mozi,Mage,Mid_Lane,Red_Win
3,4,4/11/2025,Match_20250411_1,1,Spring_2025,NMSS,Red,4,True,134.0,Dharma,Fighter,Clash_Lane,Red_Win
4,5,4/11/2025,Match_20250411_1,1,Spring_2025,RRQ,Blue,5,False,157.0,Mai_Shiranui,Assasin,Mid_Lane,Red_Win
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3361,3362,6/1/2025,Match_20250601_187,187,Spring_2025,DMT,Blue,14,True,NaN,NaN,NaN,NaN,Red_Win
3362,3363,6/1/2025,Match_20250601_187,187,Spring_2025,KAGE,Red,15,False,199.0,Arli,Marksman,Farm_Lane,Red_Win
3363,3364,6/1/2025,Match_20250601_187,187,Spring_2025,DMT,Blue,16,False,199.0,Arli,Marksman,Farm_Lane,Red_Win
3364,3365,6/1/2025,Match_20250601_187,187,Spring_2025,DMT,Blue,17,False,171.0,Zhang_Fei,Support,Roamer,Red_Win


Dataset Draft pick IKL Fall 25


,NO,Tanggal_Match,Match_ID,Match_Sequence,Tournament_Season,Team_Name,Team_Side,Draft_Order,Is_Ban,Hero_ID,Hero_Image,Hero_Name,Hero_Tipe,Hero_Role,Match_Outcome
0,1,8/22/2025,Match_20250822_1,1,Fall_2025,AE,Blue,1,True,176.0,NaN,Yuhuan,Mage,Mid_Lane,Red_Win
1,2,8/22/2025,Match_20250822_1,1,Fall_2025,VESA,Red,2,True,146.0,NaN,Luna,Fighter,Jungler,Red_Win
2,3,8/22/2025,Match_20250822_1,1,Fall_2025,AE,Blue,3,True,134.0,NaN,Dharma,Fighter,Clash_Lane,Red_Win
3,4,8/22/2025,Match_20250822_1,1,Fall_2025,VESA,Red,4,True,156.0,NaN,Liang,Mage,Mid_Lane,Red_Win
4,5,8/22/2025,Match_20250822_1,1,Fall_2025,AE,Blue,5,False,502.0,NaN,Pei,Assasin,Jungler,Red_Win
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3181,3182,10/5/2025,Match_20251005_177,177,Fall_2025,BTR,Blue,14,True,NaN,NaN,NaN,NaN,NaN,Blue_Win
3182,3183,10/5/2025,Match_20251005_177,177,Fall_2025,KAGE,Red,15,False,199.0,NaN,Arli,Marksman,Farm_Lane,Blue_Win
3183,3184,10/5/2025,Match_20251005_177,177,Fall_2025,BTR,Blue,16,False,519.0,NaN,Ao_Yin,Marksman,Farm_Lane,Blue_Win
3184,3185,10/5/2025,Match_20251005_177,177,Fall_2025,BTR,Blue,17,False,159.0,NaN,Dolia,Support,Roamer,Blue_Win


Dataset Draft pick IKL Spring 26


,NO,Tanggal_Match,Match_ID,Match_Sequence,Tournament_Season,Team_Name,Team_Side,Draft_Order,Is_Ban,Hero_ID,Hero_Image,Hero_Name,Hero_Tipe,Hero_Role,Match_Outcome
0,1,2/27/2026,Match_20260227_1,1,Spring_2026,ONIC,Blue,1,True,172.0,NaN,Chicha,Fighter,Clash_Lane,Blue_Win
1,2,2/27/2026,Match_20260227_1,1,Spring_2026,SE,Red,2,True,140.0,NaN,Guan_Yu,Fighter,Clash_Lane,Blue_Win
2,3,2/27/2026,Match_20260227_1,1,Spring_2026,ONIC,Blue,3,True,159.0,NaN,Dolia,Support,Roamer,Blue_Win
3,4,2/27/2026,Match_20260227_1,1,Spring_2026,SE,Red,4,True,134.0,NaN,Dharma,Fighter,Clash_Lane,Blue_Win
4,5,2/27/2026,Match_20260227_1,1,Spring_2026,ONIC,Blue,5,False,521.0,NaN,Haya,Mage,Mid_Lane,Blue_Win
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3685,3686,4/12/2026,Match_20260412_205,205,Spring_2026,BTR,Blue,14,True,111.0,NaN,Lady_Sun,Marksman,Farm_Lane,Blue_Win
3686,3687,4/12/2026,Match_20260412_205,205,Spring_2026,DMT,Red,15,False,536.0,NaN,Charlotte,Fighter,Clash_Lane,Blue_Win
3687,3688,4/12/2026,Match_20260412_205,205,Spring_2026,BTR,Blue,16,False,110.0,NaN,Garuda_Khageswara,Mage,Mid_Lane,Blue_Win
3688,3689,4/12/2026,Match_20260412_205,205,Spring_2026,BTR,Blue,17,False,128.0,NaN,Fatih,Fighter,Clash_Lane,Blue_Win


Dataset Draft pick Gabungan


,NO,Tanggal_Match,Match_ID,Match_Sequence,Tournament_Season,Team_Name,Team_Side,Draft_Order,Is_Ban,Hero_ID,Hero_Name,Hero_Tipe,Hero_Role,Match_Outcome,Hero_Image
0,1,4/11/2025,Match_20250411_1,1,Spring_2025,RRQ,Blue,1,True,564.0,Mayene,Fighter,Clash_Lane,Red_Win,NaN
1,2,4/11/2025,Match_20250411_1,1,Spring_2025,NMSS,Red,2,True,159.0,Dolia,Support,Roamer,Red_Win,NaN
2,3,4/11/2025,Match_20250411_1,1,Spring_2025,RRQ,Blue,3,True,108.0,Mozi,Mage,Mid_Lane,Red_Win,NaN
3,4,4/11/2025,Match_20250411_1,1,Spring_2025,NMSS,Red,4,True,134.0,Dharma,Fighter,Clash_Lane,Red_Win,NaN
4,5,4/11/2025,Match_20250411_1,1,Spring_2025,RRQ,Blue,5,False,157.0,Mai_Shiranui,Assasin,Mid_Lane,Red_Win,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10237,10238,4/12/2026,Match_20260412_205,569,Spring_2026,BTR,Blue,14,True,111.0,Lady_Sun,Marksman,Farm_Lane,Blue_Win,NaN
10238,10239,4/12/2026,Match_20260412_205,569,Spring_2026,DMT,Red,15,False,536.0,Charlotte,Fighter,Clash_Lane,Blue_Win,NaN
10239,10240,4/12/2026,Match_20260412_205,569,Spring_2026,BTR,Blue,16,False,110.0,Garuda_Khageswara,Mage,Mid_Lane,Blue_Win,NaN
10240,10241,4/12/2026,Match_20260412_205,569,Spring_2026,BTR,Blue,17,False,128.0,Fatih,Fighter,Clash_Lane,Blue_Win,NaN


DATASET HERO LIST


,NO,Hero_ID,Hero_Name,Hero_Tipe,Default_Role
0,1,533,Agudo,Marksman,Jungler
1,2,545,Alessio,Marksman,Farm_Lane
2,3,514,Allain,Fighter,Clash_Lane
3,4,142,Angela,Mage,Mid_Lane
4,5,519,Ao_Yin,Marksman,Farm_Lane


In [ ]:
print("="*60)
print("INFORMASI DATASET DRAFT PICK")
print("="*60)

display(df_draft.info())

print("="*60)
print("INFORMASI DATASET HERO LIST")
print("="*60)

display(df_hero.info())

INFORMASI DATASET DRAFT PICK
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10242 entries, 0 to 10241
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   NO                 10242 non-null  int64  
 1   Tanggal_Match      10242 non-null  object 
 2   Match_ID           10242 non-null  object 
 3   Match_Sequence     10242 non-null  int64  
 4   Tournament_Season  10242 non-null  object 
 5   Team_Name          10242 non-null  object 
 6   Team_Side          10242 non-null  object 
 7   Draft_Order        10242 non-null  int64  
 8   Is_Ban             10242 non-null  bool   
 9   Hero_ID            10192 non-null  float64
 10  Hero_Name          10192 non-null  object 
 11  Hero_Tipe          10192 non-null  object 
 12  Hero_Role          10192 non-null  object 
 13  Match_Outcome      10242 non-null  object 
 14  Hero_Image         0 non-null      float64
dtypes: bool(1), float64(2), int64(3), object(

None

INFORMASI DATASET HERO LIST
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113 entries, 0 to 112
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   NO            113 non-null    int64 
 1   Hero_ID       113 non-null    int64 
 2   Hero_Name     113 non-null    object
 3   Hero_Tipe     113 non-null    object
 4   Default_Role  113 non-null    object
dtypes: int64(2), object(3)
memory usage: 4.5+ KB


None

In [ ]:
# ==============================================================================
# REKAPITULASI MISSING VALUE
# ==============================================================================
print("=" * 60)
print("Missing Value")
print("=" * 60)

# 1. Tabel Rekapitulasi Draft Pick (Per Musim & Total)
df_missing_draft = (
    pd.DataFrame({
        'Spring 25': df_spring_25.isnull().sum(),
        'Fall 25': df_fall_25.isnull().sum(),
        'Spring 26': df_spring_26.isnull().sum(),
        'Total': df_draft.isnull().sum()
    })
    .fillna(0)
    .astype(int)
    .rename_axis('Nama Atribut')
    .reset_index()
)

print("Draft Pick")
display(df_missing_draft)

# 2. Tabel Rekapitulasi Hero List
df_missing_hero = (
    df_hero.isnull().sum()
    .rename_axis('Nama Atribut')
    .reset_index(name='Total')
)

print("\n" + "=" * 60)
print("Hero List")
display(df_missing_hero)

Missing Value
Draft Pick


,Nama Atribut,Spring 25,Fall 25,Spring 26,Total
0,Draft_Order,0,0,0,0
1,Hero_ID,12,16,22,50
2,Hero_Image,0,3186,3690,10242
3,Hero_Name,12,16,22,50
4,Hero_Role,12,16,22,50
5,Hero_Tipe,12,16,22,50
6,Is_Ban,0,0,0,0
7,Match_ID,0,0,0,0
8,Match_Outcome,0,0,0,0
9,Match_Sequence,0,0,0,0



Hero List


,Nama Atribut,Total
0,NO,0
1,Hero_ID,0
2,Hero_Name,0
3,Hero_Tipe,0
4,Default_Role,0


In [ ]:
print("="*60)
print("DUPLICATE DATA")
print("="*60)

print(f"Duplicate Draft Pick Spring 25 : {df_spring_25.duplicated().sum()}")
print(f"Duplicate Draft Pick Fall 25   : {df_fall_25.duplicated().sum()}")
print(f"Duplicate Draft Pick Spring 26 : {df_spring_26.duplicated().sum()}")

print(f"Duplicate Draft Pick : {df_draft.duplicated().sum()}")

print(f"Duplicate Hero List  : {df_hero.duplicated().sum()}")

DUPLICATE DATA
Duplicate Draft Pick Spring 25 : 0
Duplicate Draft Pick Fall 25   : 0
Duplicate Draft Pick Spring 26 : 0
Duplicate Draft Pick : 0
Duplicate Hero List  : 0


In [ ]:
# ==============================================================================
# 2A. MODUL VALIDASI TRIANGULASI MAKRO: PRIMER VS LIQUIPEDIA (BAB 3.1.2.a)
# ==============================================================================

# Target kuota pertandingan dan baris per musim sesuai Tabel 3.2 Proposal
TARGET_KUOTA = {
    "Spring_2025": {"matches": 187, "rows": 3366},
    "Fall_2025":   {"matches": 177, "rows": 3186},
    "Spring_2026": {"matches": 205, "rows": 3690}
}

def diagnosa_baris(row) -> str:
    """Mendiagnosa sumber human error berdasarkan pola selisih variabel."""
    tol_wr = 0.06  # Toleransi pembulatan desimal Win Rate di Liquipedia
    if (
        row["Diff_Picks"] == 0 and
        row["Diff_Wins"] == 0 and
        row["Diff_Blue_Picks"] == 0 and
        row["Diff_Blue_Wins"] == 0 and
        row["Diff_Red_Picks"] == 0 and
        row["Diff_Red_Wins"] == 0 and
        row["Diff_Bans"] == 0 and
        abs(row["Diff_WR"]) <= tol_wr
    ):
        return "VALID"

    issues = []
    if row["Primer_Contest"] > 0 and row["LP_Contest"] == 0:
        issues.append("HANYA ADA DI PRIMER (Cek Ejaan Nama Hero)")
    elif row["Primer_Contest"] == 0 and row["LP_Contest"] > 0:
        issues.append("HANYA ADA DI LP (Cek HERO_NAME_MAP)")
    else:
        if row["Diff_Contest"] == 0 and (row["Diff_Picks"] != 0 or row["Diff_Bans"] != 0):
            issues.append("TERTUKAR PICK/BAN (Cek Is_Ban)")
        if row["Diff_Picks"] == 0 and (row["Diff_Blue_Picks"] != 0 or row["Diff_Red_Picks"] != 0):
            issues.append("TERTUKAR SISI TIM (Cek Team_Side Blue/Red)")
        if row["Diff_Picks"] == 0 and row["Diff_Wins"] != 0:
            issues.append("SALAH OUTCOME (Cek Match_Outcome)")
        if row["Diff_Contest"] != 0:
            issues.append("SELISIH KUANTITAS (Cek Typo Hero / Anomali Match)")

    return " | ".join(issues) if issues else "SELISIH PEMBULATAN WR"


def resolve_data_path(filepath: str, default_dir: str = "../Data") -> str:
    """Mencari file secara otomatis di folder saat ini atau di dalam ../Data/."""
    if os.path.exists(filepath):
        return filepath
    alt_path = os.path.join(default_dir, os.path.basename(filepath))
    return alt_path


def validasi_triangulasi_musim(
    df_primer_musim: pd.DataFrame,
    file_lp_csv: str,
    season_label: str,
    file_lp_summary_csv: str = None,
    output_dir: str = "../Data"
) -> pd.DataFrame:
    """
    Menghasilkan output perbandingan makro tingkat musim (Games Played, Blue W-L, Red W-L)
    serta mengembalikan DataFrame komparasi lengkap tingkat Hero (df_comp).
    """
    df_p = df_primer_musim.copy()
    if file_lp_summary_csv is None:
        file_lp_summary_csv = f"Liquipedia_Ringkasan_{season_label}.csv"

    # Otomatis arahkan ke ../Data/ jika file hanya ditulis nama filenya saja
    path_lp_csv     = resolve_data_path(file_lp_csv, output_dir)
    path_lp_summary = resolve_data_path(file_lp_summary_csv, output_dir)

    # =========================================================================
    # 1. AGREGASI LEVEL PERTANDINGAN (MATCH-LEVEL) DARI DATA PRIMER
    # =========================================================================
    if "Match_ID" in df_p.columns:
        df_matches = df_p.drop_duplicates(subset=["Match_ID"]).copy()
    elif "Match_Sequence" in df_p.columns:
        df_matches = df_p.drop_duplicates(subset=["Match_Sequence"]).copy()
    elif "Draft_Order" in df_p.columns:
        df_matches = df_p[df_p["Draft_Order"] == 1].copy()
    else:
        df_matches = df_p.iloc[::18].copy()

    outcomes_clean     = df_matches["Match_Outcome"].astype(str).str.strip().str.lower()
    total_matches      = len(df_matches)
    primer_blue_wins   = int((outcomes_clean == "blue_win").sum())
    primer_red_wins    = int((outcomes_clean == "red_win").sum())
    primer_blue_losses = primer_red_wins
    primer_red_losses  = primer_blue_wins

    primer_blue_wr = round((primer_blue_wins / total_matches) * 100, 2) if total_matches > 0 else 0.0
    primer_red_wr  = round((primer_red_wins / total_matches) * 100, 2) if total_matches > 0 else 0.0

    # Muat data ringkasan musim dari Liquipedia
    df_lp_sum       = pd.read_csv(path_lp_summary).iloc[0]
    lp_games_played = int(df_lp_sum["LP_Games_Played"])
    lp_blue_wins    = int(df_lp_sum["LP_Blue_Wins"])
    lp_blue_losses  = int(df_lp_sum["LP_Blue_Losses"])
    lp_blue_wr      = float(df_lp_sum["LP_Blue_WR"])
    lp_red_wins     = int(df_lp_sum["LP_Red_Wins"])
    lp_red_losses   = int(df_lp_sum["LP_Red_Losses"])
    lp_red_wr       = float(df_lp_sum["LP_Red_WR"])

    target_info    = TARGET_KUOTA.get(season_label, {})
    target_matches = target_info.get("matches", lp_games_played)
    target_rows    = target_info.get("rows", lp_games_played * 18)

    df_ringkasan_makro = pd.DataFrame([
        {
            "Variabel Makro Musim": "Total Pertandingan (Games Played)",
            "Data Primer": f"{total_matches}",
            "Data Liquipedia": f"{lp_games_played} (Target: {target_matches})",
            "Selisih (Diff)": f"{total_matches - lp_games_played:+d}" if (total_matches - lp_games_played) != 0 else "0",
            "Status": "VALID" if total_matches == lp_games_played else "CEK_MATCH"
        },
        {
            "Variabel Makro Musim": "Total Baris Aksi Draft (18x Match)",
            "Data Primer": f"{len(df_p)}",
            "Data Liquipedia": f"{lp_games_played * 18} (Target: {target_rows})",
            "Selisih (Diff)": f"{len(df_p) - (lp_games_played * 18):+d}" if (len(df_p) - (lp_games_played * 18)) != 0 else "0",
            "Status": "VALID" if len(df_p) == (lp_games_played * 18) else "CEK_BARIS"
        },
        {
            "Variabel Makro Musim": "Blue Side - Total Wins (W)",
            "Data Primer": f"{primer_blue_wins}",
            "Data Liquipedia": f"{lp_blue_wins}",
            "Selisih (Diff)": f"{primer_blue_wins - lp_blue_wins:+d}" if (primer_blue_wins - lp_blue_wins) != 0 else "0",
            "Status": "VALID" if primer_blue_wins == lp_blue_wins else "CEK_OUTCOME"
        },
        {
            "Variabel Makro Musim": "Blue Side - Total Losses (L)",
            "Data Primer": f"{primer_blue_losses}",
            "Data Liquipedia": f"{lp_blue_losses}",
            "Selisih (Diff)": f"{primer_blue_losses - lp_blue_losses:+d}" if (primer_blue_losses - lp_blue_losses) != 0 else "0",
            "Status": "VALID" if primer_blue_losses == lp_blue_losses else "CEK_OUTCOME"
        },
        {
            "Variabel Makro Musim": "Blue Side - Win Rate (WR%)",
            "Data Primer": f"{primer_blue_wr:.2f}%",
            "Data Liquipedia": f"{lp_blue_wr:.2f}%",
            "Selisih (Diff)": f"{primer_blue_wr - lp_blue_wr:+.2f}%" if abs(primer_blue_wr - lp_blue_wr) > 0.05 else "0.00%",
            "Status": "VALID" if abs(primer_blue_wr - lp_blue_wr) <= 0.05 else "CEK_OUTCOME"
        },
        {
            "Variabel Makro Musim": "Red Side - Total Wins (W)",
            "Data Primer": f"{primer_red_wins}",
            "Data Liquipedia": f"{lp_red_wins}",
            "Selisih (Diff)": f"{primer_red_wins - lp_red_wins:+d}" if (primer_red_wins - lp_red_wins) != 0 else "0",
            "Status": "VALID" if primer_red_wins == lp_red_wins else "CEK_OUTCOME"
        },
        {
            "Variabel Makro Musim": "Red Side - Total Losses (L)",
            "Data Primer": f"{primer_red_losses}",
            "Data Liquipedia": f"{lp_red_losses}",
            "Selisih (Diff)": f"{primer_red_losses - lp_red_losses:+d}" if (primer_red_losses - lp_red_losses) != 0 else "0",
            "Status": "VALID" if primer_red_losses == lp_red_losses else "CEK_OUTCOME"
        },
        {
            "Variabel Makro Musim": "Red Side - Win Rate (WR%)",
            "Data Primer": f"{primer_red_wr:.2f}%",
            "Data Liquipedia": f"{lp_red_wr:.2f}%",
            "Selisih (Diff)": f"{primer_red_wr - lp_red_wr:+.2f}%" if abs(primer_red_wr - lp_red_wr) > 0.05 else "0.00%",
            "Status": "VALID" if abs(primer_red_wr - lp_red_wr) <= 0.05 else "CEK_OUTCOME"
        },
        {
            "Variabel Makro Musim": "Rekapitulasi Sisi (Blue | Red)",
            "Data Primer": f"{primer_blue_wins}W-{primer_blue_losses}L ({primer_blue_wr:.2f}%) | {primer_red_wins}W-{primer_red_losses}L ({primer_red_wr:.2f}%)",
            "Data Liquipedia": f"{lp_blue_wins}W-{lp_blue_losses}L ({lp_blue_wr:.2f}%) | {lp_red_wins}W-{lp_red_losses}L ({lp_red_wr:.2f}%)",
            "Selisih (Diff)": "IDENTIK" if (primer_blue_wins == lp_blue_wins and primer_red_wins == lp_red_wins) else "BERSELISIH",
            "Status": "VALID" if (primer_blue_wins == lp_blue_wins and primer_red_wins == lp_red_wins) else "CEK_OUTCOME"
        }
    ])

    # =========================================================================
    # 2. AGREGASI LEVEL HERO DARI DATA PRIMER
    # =========================================================================
    df_p_valid = df_p.dropna(subset=["Hero_Name"]).copy()
    df_p_valid["Hero_Name"] = df_p_valid["Hero_Name"].astype(str).str.strip()

    side_clean    = df_p_valid["Team_Side"].astype(str).str.strip().str.lower()
    outcome_clean = df_p_valid["Match_Outcome"].astype(str).str.strip().str.lower()

    if "Is_Win" not in df_p_valid.columns:
        df_p_valid["Is_Win"] = (
            ((side_clean == "blue") & (outcome_clean == "blue_win")) |
            ((side_clean == "red") & (outcome_clean == "red_win"))
        )

    df_picks = df_p_valid[df_p_valid["Is_Ban"] == False]
    df_bans  = df_p_valid[df_p_valid["Is_Ban"] == True]
    df_blue  = df_picks[df_picks["Team_Side"].astype(str).str.strip().str.lower() == "blue"]
    df_red   = df_picks[df_picks["Team_Side"].astype(str).str.strip().str.lower() == "red"]

    agg_total = df_picks.groupby("Hero_Name").agg(Primer_Picks=("Hero_Name", "count"), Primer_Wins=("Is_Win", "sum"))
    agg_blue  = df_blue.groupby("Hero_Name").agg(Primer_Blue_Picks=("Hero_Name", "count"), Primer_Blue_Wins=("Is_Win", "sum"))
    agg_red   = df_red.groupby("Hero_Name").agg(Primer_Red_Picks=("Hero_Name", "count"), Primer_Red_Wins=("Is_Win", "sum"))
    agg_bans  = df_bans.groupby("Hero_Name").agg(Primer_Bans=("Hero_Name", "count"))

    df_agg_primer = pd.concat([agg_total, agg_blue, agg_red, agg_bans], axis=1).fillna(0).reset_index()
    df_agg_primer["Primer_Contest"] = df_agg_primer["Primer_Picks"] + df_agg_primer["Primer_Bans"]

    df_agg_primer["Primer_WR"] = (
        (df_agg_primer["Primer_Wins"] / df_agg_primer["Primer_Picks"].replace(0, np.nan)) * 100
    ).fillna(0.0).round(2)

    df_agg_primer["Primer_Blue_WR"] = (
        (df_agg_primer["Primer_Blue_Wins"] / df_agg_primer["Primer_Blue_Picks"].replace(0, np.nan)) * 100
    ).fillna(0.0).round(2)

    df_agg_primer["Primer_Red_WR"] = (
        (df_agg_primer["Primer_Red_Wins"] / df_agg_primer["Primer_Red_Picks"].replace(0, np.nan)) * 100
    ).fillna(0.0).round(2)

    df_agg_primer["Primer_Contest_Rate"] = (
        (df_agg_primer["Primer_Contest"] / total_matches) * 100 if total_matches > 0 else 0.0
    ).round(2)

    # Muat Data Liquipedia musim terkait dan gabungkan
    df_lp = pd.read_csv(path_lp_csv)
    df_comp = pd.merge(df_agg_primer, df_lp, on="Hero_Name", how="outer").fillna(0)
    df_comp["Tournament_Season"] = season_label

    int_cols_all = [
        "Primer_Picks", "Primer_Wins", "Primer_Blue_Picks", "Primer_Blue_Wins",
        "Primer_Red_Picks", "Primer_Red_Wins", "Primer_Bans", "Primer_Contest",
        "LP_Picks", "LP_Wins", "LP_Blue_Picks", "LP_Blue_Wins",
        "LP_Red_Picks", "LP_Red_Wins", "LP_Bans", "LP_Contest"
    ]
    for col in int_cols_all:
        df_comp[col] = df_comp[col].astype(int)

    df_comp["Diff_Picks"]        = df_comp["Primer_Picks"] - df_comp["LP_Picks"]
    df_comp["Diff_Wins"]         = df_comp["Primer_Wins"] - df_comp["LP_Wins"]
    df_comp["Diff_WR"]           = (df_comp["Primer_WR"] - df_comp["LP_WR"]).round(2)

    df_comp["Diff_Blue_Picks"]   = df_comp["Primer_Blue_Picks"] - df_comp["LP_Blue_Picks"]
    df_comp["Diff_Blue_Wins"]    = df_comp["Primer_Blue_Wins"] - df_comp["LP_Blue_Wins"]
    df_comp["Diff_Blue_WR"]      = (df_comp["Primer_Blue_WR"] - df_comp["LP_Blue_WR"]).round(2)

    df_comp["Diff_Red_Picks"]    = df_comp["Primer_Red_Picks"] - df_comp["LP_Red_Picks"]
    df_comp["Diff_Red_Wins"]     = df_comp["Primer_Red_Wins"] - df_comp["LP_Red_Wins"]
    df_comp["Diff_Red_WR"]       = (df_comp["Primer_Red_WR"] - df_comp["LP_Red_WR"]).round(2)

    df_comp["Diff_Bans"]         = df_comp["Primer_Bans"] - df_comp["LP_Bans"]
    df_comp["Diff_Contest"]      = df_comp["Primer_Contest"] - df_comp["LP_Contest"]
    df_comp["Diff_Contest_Rate"] = (df_comp["Primer_Contest_Rate"] - df_comp["LP_Contest_Rate"]).round(2)

    df_comp["Diagnosa_Otomatis"] = df_comp.apply(diagnosa_baris, axis=1)

    ordered_cols = [
        "Tournament_Season", "Hero_Name", "Diagnosa_Otomatis",
        "Primer_Picks", "LP_Picks", "Diff_Picks",
        "Primer_Wins", "LP_Wins", "Diff_Wins",
        "Primer_WR", "LP_WR", "Diff_WR",
        "Primer_Blue_Picks", "LP_Blue_Picks", "Diff_Blue_Picks",
        "Primer_Blue_Wins", "LP_Blue_Wins", "Diff_Blue_Wins",
        "Primer_Blue_WR", "LP_Blue_WR", "Diff_Blue_WR",
        "Primer_Red_Picks", "LP_Red_Picks", "Diff_Red_Picks",
        "Primer_Red_Wins", "LP_Red_Wins", "Diff_Red_Wins",
        "Primer_Red_WR", "LP_Red_WR", "Diff_Red_WR",
        "Primer_Bans", "LP_Bans", "Diff_Bans",
        "Primer_Contest", "LP_Contest", "Diff_Contest",
        "Primer_Contest_Rate", "LP_Contest_Rate", "Diff_Contest_Rate"
    ]
    df_comp = df_comp[ordered_cols]

    df_comp["Is_Valid_Sort"] = df_comp["Diagnosa_Otomatis"] == "VALID"
    df_comp = df_comp.sort_values(
        by=["Is_Valid_Sort", "LP_Picks"], ascending=[True, False]
    ).drop(columns=["Is_Valid_Sort"]).reset_index(drop=True)

    # Simpan ke CSV di folder ../Data/ (jika ada) atau direktori saat ini
    save_dir = output_dir if os.path.exists(output_dir) else "."
    output_filename = os.path.join(save_dir, f"Hasil_Triangulasi_{season_label}.csv")
    output_sum_name = os.path.join(save_dir, f"Ringkasan_Makro_{season_label}.csv")
    df_comp.to_csv(output_filename, index=False)
    df_ringkasan_makro.to_csv(output_sum_name, index=False)

    jml_valid   = (df_comp["Diagnosa_Otomatis"] == "VALID").sum()
    jml_anomali = len(df_comp) - jml_valid

    print("=" * 98)
    print(f"RINGKASAN TRIANGULASI MAKRO - MUSIM: {season_label}")
    print("=" * 98)
    print(df_ringkasan_makro.to_string(index=False))
    print("-" * 98)
    print(f"Total Hero Tercatat : {len(df_comp)} Hero (Primer: {len(df_agg_primer)} Hero | Liquipedia: {len(df_lp)} Hero)")
    print(f"Hero Cocok 100%     : {jml_valid} Hero ({(jml_valid/len(df_comp)*100):.2f}%)")
    print(f"Hero Berselisih     : {jml_anomali} Hero")
    print(f"File Output Tabel   : '{output_filename}' & '{output_sum_name}'")

    unmatched = df_comp[(df_comp["Primer_Contest"] == 0) | (df_comp["LP_Contest"] == 0)]
    if not unmatched.empty:
        print("\n[PERINGATAN] Ditemukan Hero yang hanya muncul di salah satu sumber (Cek ejaan / HERO_NAME_MAP):")
        print(unmatched[["Hero_Name", "Primer_Picks", "Primer_Bans", "LP_Picks", "LP_Bans", "Diagnosa_Otomatis"]].to_string(index=False))
    print("\n")

    return df_comp

In [ ]:
# Validasi Spring 2025 (Menggunakan df_spring_25 yang sudah dimuat di Sel 2)
hasil_s25 = validasi_triangulasi_musim(df_spring_25, "Liquipedia_Spring_2025.csv", "Spring_2025")

RINGKASAN TRIANGULASI MAKRO - MUSIM: Spring_2025
              Variabel Makro Musim                           Data Primer                       Data Liquipedia Selisih (Diff)      Status
 Total Pertandingan (Games Played)                                   187                     187 (Target: 187)              0       VALID
Total Baris Aksi Draft (18x Match)                                  3366                   3366 (Target: 3366)              0       VALID
        Blue Side - Total Wins (W)                                    84                                    85             -1 CEK_OUTCOME
      Blue Side - Total Losses (L)                                   103                                   102             +1 CEK_OUTCOME
        Blue Side - Win Rate (WR%)                                44.92%                                45.45%         -0.53% CEK_OUTCOME
         Red Side - Total Wins (W)                                   103                                   102             

In [ ]:
# Validasi Fall 2025
hasil_f25 = validasi_triangulasi_musim(df_fall_25, "Liquipedia_Fall_2025.csv", "Fall_2025")

RINGKASAN TRIANGULASI MAKRO - MUSIM: Fall_2025
              Variabel Makro Musim                         Data Primer                     Data Liquipedia Selisih (Diff)      Status
 Total Pertandingan (Games Played)                                 177                   177 (Target: 177)              0       VALID
Total Baris Aksi Draft (18x Match)                                3186                 3186 (Target: 3186)              0       VALID
        Blue Side - Total Wins (W)                                  89                                  87             +2 CEK_OUTCOME
      Blue Side - Total Losses (L)                                  88                                  90             -2 CEK_OUTCOME
        Blue Side - Win Rate (WR%)                              50.28%                              49.15%         +1.13% CEK_OUTCOME
         Red Side - Total Wins (W)                                  88                                  90             -2 CEK_OUTCOME
       Red Side

In [ ]:
# Validasi Spring 2026
hasil_s26 = validasi_triangulasi_musim(df_spring_26, "Liquipedia_Spring_2026.csv", "Spring_2026")

RINGKASAN TRIANGULASI MAKRO - MUSIM: Spring_2026
              Variabel Makro Musim                           Data Primer                       Data Liquipedia Selisih (Diff)      Status
 Total Pertandingan (Games Played)                                   205                     205 (Target: 205)              0       VALID
Total Baris Aksi Draft (18x Match)                                  3690                   3690 (Target: 3690)              0       VALID
        Blue Side - Total Wins (W)                                   120                                   120              0       VALID
      Blue Side - Total Losses (L)                                    85                                    84             +1 CEK_OUTCOME
        Blue Side - Win Rate (WR%)                                58.54%                                58.54%          0.00%       VALID
         Red Side - Total Wins (W)                                    85                                    84             

In [ ]:
kolom_ringkas_naskah = [
    "Hero_Name", "Diagnosa_Otomatis",
    "Primer_Picks", "LP_Picks", "Diff_Picks",
    "Primer_Wins", "LP_Wins", "Diff_Wins",
    "Primer_WR", "LP_WR",
    "Primer_Blue_Picks", "LP_Blue_Picks", "Diff_Blue_Picks",
    "Primer_Red_Picks", "LP_Red_Picks", "Diff_Red_Picks",
    "Primer_Bans", "LP_Bans", "Diff_Bans",
    "Primer_Contest", "LP_Contest", "Diff_Contest"
]

In [ ]:
display(hasil_s25[kolom_ringkas_naskah])

,Hero_Name,Diagnosa_Otomatis,Primer_Picks,LP_Picks,Diff_Picks,Primer_Wins,LP_Wins,Diff_Wins,Primer_WR,LP_WR,...,Diff_Blue_Picks,Primer_Red_Picks,LP_Red_Picks,Diff_Red_Picks,Primer_Bans,LP_Bans,Diff_Bans,Primer_Contest,LP_Contest,Diff_Contest
0,Ao_Yin,TERTUKAR SISI TIM (Cek Team_Side Blue/Red) | S...,92,92,0,41,42,-1,44.57,45.65,...,1,50,51,-1,38,38,0,130,130,0
1,Zhang_Fei,TERTUKAR SISI TIM (Cek Team_Side Blue/Red),79,79,0,45,45,0,56.96,56.96,...,-1,47,46,1,69,69,0,148,148,0
2,Arli,TERTUKAR PICK/BAN (Cek Is_Ban),72,71,1,34,33,1,47.22,46.48,...,1,21,21,0,100,101,-1,172,172,0
3,Yixing,TERTUKAR SISI TIM (Cek Team_Side Blue/Red) | S...,70,70,0,32,31,1,45.71,44.29,...,1,40,41,-1,55,55,0,125,125,0
4,Menki,SELISIH KUANTITAS (Cek Typo Hero / Anomali Match),67,67,0,34,34,0,50.75,50.75,...,0,32,32,0,42,41,1,109,108,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,Wukong,VALID,1,1,0,0,0,0,0.00,0.00,...,0,1,1,0,0,0,0,1,1,0
75,Xiao_Qiao,VALID,1,1,0,0,0,0,0.00,0.00,...,0,1,1,0,0,0,0,1,1,0
76,Yao,VALID,1,1,0,0,0,0,0.00,0.00,...,0,0,0,0,0,0,0,1,1,0
77,Zhou_Yu,VALID,1,1,0,1,1,0,100.00,100.00,...,0,1,1,0,0,0,0,1,1,0


In [ ]:
display(hasil_f25[kolom_ringkas_naskah])

,Hero_Name,Diagnosa_Otomatis,Primer_Picks,LP_Picks,Diff_Picks,Primer_Wins,LP_Wins,Diff_Wins,Primer_WR,LP_WR,...,Diff_Blue_Picks,Primer_Red_Picks,LP_Red_Picks,Diff_Red_Picks,Primer_Bans,LP_Bans,Diff_Bans,Primer_Contest,LP_Contest,Diff_Contest
0,Yuhuan,TERTUKAR SISI TIM (Cek Team_Side Blue/Red) | S...,91,91,0,54,54,0,59.34,59.34,...,-1,46,45,1,57,56,1,148,147,1
1,Ao_Yin,SELISIH KUANTITAS (Cek Typo Hero / Anomali Match),74,77,-3,40,40,0,54.05,51.95,...,-3,38,38,0,40,40,0,114,117,-3
2,Arli,SELISIH KUANTITAS (Cek Typo Hero / Anomali Match),68,71,-3,32,35,-3,47.06,49.30,...,-3,32,32,0,88,87,1,156,158,-2
3,Pei,SELISIH KUANTITAS (Cek Typo Hero / Anomali Match),67,66,1,35,34,1,52.24,51.52,...,2,33,34,-1,76,76,0,143,142,1
4,Zhang_Fei,SELISIH KUANTITAS (Cek Typo Hero / Anomali Match),66,65,1,37,36,1,56.06,55.38,...,2,32,33,-1,40,40,0,106,105,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,Fang,VALID,1,1,0,0,0,0,0.00,0.00,...,0,1,1,0,0,0,0,1,1,0
70,Garo,VALID,1,1,0,0,0,0,0.00,0.00,...,0,0,0,0,1,1,0,2,2,0
71,Han_Xin,VALID,1,1,0,0,0,0,0.00,0.00,...,0,1,1,0,0,0,0,1,1,0
72,Kui,VALID,1,1,0,1,1,0,100.00,100.00,...,0,1,1,0,0,0,0,1,1,0


In [ ]:
display(hasil_s26[kolom_ringkas_naskah])

,Hero_Name,Diagnosa_Otomatis,Primer_Picks,LP_Picks,Diff_Picks,Primer_Wins,LP_Wins,Diff_Wins,Primer_WR,LP_WR,...,Diff_Blue_Picks,Primer_Red_Picks,LP_Red_Picks,Diff_Red_Picks,Primer_Bans,LP_Bans,Diff_Bans,Primer_Contest,LP_Contest,Diff_Contest
0,Zhang_Fei,TERTUKAR SISI TIM (Cek Team_Side Blue/Red),81,81,0,41,41,0,50.62,50.62,...,-1,46,45,1,51,51,0,132,132,0
1,Ao_Yin,TERTUKAR PICK/BAN (Cek Is_Ban),74,73,1,36,35,1,48.65,47.95,...,1,38,38,0,85,86,-1,159,159,0
2,Chano,SELISIH KUANTITAS (Cek Typo Hero / Anomali Match),73,72,1,31,31,0,42.47,43.06,...,0,39,38,1,18,18,0,91,90,1
3,Dharma,TERTUKAR SISI TIM (Cek Team_Side Blue/Red) | S...,71,71,0,37,37,0,52.11,52.11,...,1,33,34,-1,70,71,-1,141,142,-1
4,Mai_Shiranui,SELISIH KUANTITAS (Cek Typo Hero / Anomali Match),62,62,0,33,33,0,53.23,53.23,...,0,19,19,0,78,79,-1,140,141,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,Liu_Shan,VALID,1,1,0,1,1,0,100.00,100.00,...,0,0,0,0,0,0,0,1,1,0
80,Milady,VALID,1,1,0,1,1,0,100.00,100.00,...,0,1,1,0,0,0,0,1,1,0
81,Shouyue,VALID,1,1,0,1,1,0,100.00,100.00,...,0,0,0,0,0,0,0,1,1,0
82,Xiang_Yu,VALID,1,1,0,0,0,0,0.00,0.00,...,0,0,0,0,0,0,0,1,1,0


In [ ]:
# ==============================================================================
# 2B. RANDOM SAMPLING 5% UNTUK VALIDASI MANUAL
# BAB 3.1.2.b & BAB 3.1.2.c
# ==============================================================================
random.seed(42)

MUSIM_VALIDASI = ['Spring_2025', 'Fall_2025', 'Spring_2026']
sampel_validasi = []

print("=" * 70)
print("RANDOM SAMPLING 5% PER MUSIM")
print("=" * 70)

for musim in MUSIM_VALIDASI:
    match_ids = df_draft.loc[df_draft['Tournament_Season'] == musim, 'Match_ID'].dropna().unique().tolist()
    n_sample = max(1, round(len(match_ids) * 0.05))
    selected_matches = random.sample(match_ids, n_sample)

    sampel_validasi.extend({'Tournament_Season': musim, 'Match_ID': m_id} for m_id in selected_matches)
    print(f"{musim:<12} : {n_sample} sampel dari {len(match_ids)} pertandingan")

df_sampel_validasi = pd.DataFrame(sampel_validasi)
df_sampel_validasi.insert(0, 'No', range(1, len(df_sampel_validasi) + 1))

print("\n" + "=" * 70)
print("DAFTAR SAMPEL VALIDASI MANUAL")
print("=" * 70)
display(df_sampel_validasi)

RANDOM SAMPLING 5% PER MUSIM
Spring_2025  : 9 sampel dari 187 pertandingan
Fall_2025    : 9 sampel dari 177 pertandingan
Spring_2026  : 10 sampel dari 205 pertandingan

DAFTAR SAMPEL VALIDASI MANUAL


,No,Tournament_Season,Match_ID
0,1,Spring_2025,Match_20250530_164
1,2,Spring_2025,Match_20250418_29
2,3,Spring_2025,Match_20250411_7
3,4,Spring_2025,Match_20250426_71
4,5,Spring_2025,Match_20250425_63
5,6,Spring_2025,Match_20250425_58
6,7,Spring_2025,Match_20250418_36
7,8,Spring_2025,Match_20250413_27
8,9,Spring_2025,Match_20250531_174
9,10,Fall_2025,Match_20250927_140


In [ ]:
# ==============================================================================
# 2C. PEMBENTUKAN DATAFRAME VALIDASI MIKRO 
# HASIL RANDOM SAMPLING 5% (BAB 3.1.2.b)
# ==============================================================================
SEATING_ORDER = ['Clash_Lane', 'Jungler', 'Mid_Lane', 'Farm_Lane', 'Roamer']

# 1. Pre-filter & set Categorical satu kali di awal
df_sub = df_draft[df_draft['Match_ID'].isin(df_sampel_validasi['Match_ID'])].copy()
df_sub['Role_Order'] = pd.Categorical(df_sub['Hero_Role'], categories=SEATING_ORDER, ordered=True)
match_groups = dict(tuple(df_sub.groupby('Match_ID', sort=False)))

# 2. Ekstraksi data per pertandingan
data_validasi = []
for idx, row in enumerate(df_sampel_validasi.itertuples(index=False), start=1):
    df_m = match_groups[row.Match_ID]

    # Identitas tim & pemenang
    teams = {
        side: df_m.loc[df_m['Team_Side'] == side, 'Team_Name'].dropna().iloc[0]
        for side in ('Blue', 'Red')
    }
    outcome = df_m['Match_Outcome'].dropna().iloc[0]
    winner = {'Blue_Win': teams['Blue'], 'Red_Win': teams['Red']}.get(outcome)

    # Pick (urut Seating Order) & Ban (urut Draft Order)
    df_pick = df_m[df_m['Is_Ban'] == False].sort_values(['Team_Side', 'Role_Order'])
    df_ban = df_m[df_m['Is_Ban'] == True].sort_values('Draft_Order')

    data_validasi.append({
        # Identitas Sampel
        'No': idx,
        'Tournament_Season': row.Tournament_Season,
        'Match_ID': row.Match_ID,

        # Data Primer (Otomatis dari Dataset - Read Only)
        'Blue_Team_Primer': teams['Blue'],
        'Red_Team_Primer': teams['Red'],
        'Pick_List_Blue_Primer': df_pick.loc[df_pick['Team_Side'] == 'Blue', 'Hero_Name'].dropna().tolist(),
        'Ban_List_Blue_Primer': df_ban.loc[df_ban['Team_Side'] == 'Blue', 'Hero_Name'].dropna().tolist(),
        'Pick_List_Red_Primer': df_pick.loc[df_pick['Team_Side'] == 'Red', 'Hero_Name'].dropna().tolist(),
        'Ban_List_Red_Primer': df_ban.loc[df_ban['Team_Side'] == 'Red', 'Hero_Name'].dropna().tolist(),
        'Winner_Primer': winner,

        # Data Sekunder Liquipedia & Evaluasi (Diisi Manual)
        'Blue_Team_Liquipedia': '',
        'Red_Team_Liquipedia': '',
        'Pick_List_Blue_Liquipedia': '',
        'Ban_List_Blue_Liquipedia': '',
        'Pick_List_Red_Liquipedia': '',
        'Ban_List_Red_Liquipedia': '',
        'Winner_Liquipedia': '',
        'Link_Liquipedia': '',
        'Status_Validasi': '',
        'Catatan': ''
    })

df_validasi_mikro = pd.DataFrame(data_validasi)

# 3. Tampilkan hasil
print("=" * 90)
print("DATAFRAME VALIDASI MIKRO - HASIL RANDOM SAMPLING 5%")
print("=" * 90)
print(f"Seating Order Pick  : {' -> '.join(SEATING_ORDER)}")
print(f"Jumlah pertandingan : {len(df_validasi_mikro)}")
display(df_validasi_mikro.head(5))

# 4. Export ke CSV (konversi kolom list Primer ke JSON string secara on-the-fly)
list_columns = [
    'Pick_List_Blue_Primer',
    'Ban_List_Blue_Primer',
    'Pick_List_Red_Primer',
    'Ban_List_Red_Primer'
]
output_file = "../Output/Validasi_Mikro_Sampling_5_Persen.csv"
os.makedirs(os.path.dirname(output_file), exist_ok=True)

df_validasi_mikro.assign(
    **{col: df_validasi_mikro[col].apply(json.dumps) for col in list_columns}
).to_csv(output_file, index=False, encoding='utf-8-sig')

print("\n" + "=" * 90)
print("EXPORT DATA VALIDASI MIKRO")
print("=" * 90)
print(f"File berhasil disimpan : {output_file}")

DATAFRAME VALIDASI MIKRO - HASIL RANDOM SAMPLING 5%
Seating Order Pick  : Clash_Lane -> Jungler -> Mid_Lane -> Farm_Lane -> Roamer
Jumlah pertandingan : 28


,No,Tournament_Season,Match_ID,Blue_Team_Primer,Red_Team_Primer,Pick_List_Blue_Primer,Ban_List_Blue_Primer,Pick_List_Red_Primer,Ban_List_Red_Primer,Winner_Primer,Blue_Team_Liquipedia,Red_Team_Liquipedia,Pick_List_Blue_Liquipedia,Ban_List_Blue_Liquipedia,Pick_List_Red_Liquipedia,Ban_List_Red_Liquipedia,Winner_Liquipedia,Link_Liquipedia,Status_Validasi,Catatan
0,1,Spring_2025,Match_20250530_164,RRQ,DMT,"[Dharma, Menki, Shangguan, Ao_Yin, Dolia]","[Mayene, Liang, Lady_Zhen, Augran]","[Charlotte, Lam, Yuhuan, Di_Renjie, Zhang_Fei]","[Mai_Shiranui, Arli, Marco_Polo, Chano]",DMT,,,,,,,,,,
1,2,Spring_2025,Match_20250418_29,NMSS,DMT,"[Charlotte, Menki, Lady_Zhen, Chano, Dyadia]","[Dolia, Dharma, Mozi, Angela]","[Mayene, Augran, Gan_And_Mo, Ao_Yin, Zhang_Fei]","[Mai_Shiranui, Arli, Biron, Liu_Bang]",DMT,,,,,,,,,,
2,3,Spring_2025,Match_20250411_7,TLN,BTR,"[Sun_Ce, Dian_Wei, Yuhuan, Ao_Yin, Mozi]","[Dyadia, Dharma, Nuwa, Mayene]","[Guan_Yu, Menki, Yixing, Lady_Sun, Zhang_Fei]","[Arli, Mai_Shiranui, Wang_Zhaojun, Gan_And_Mo]",TLN,,,,,,,,,,
3,4,Spring_2025,Match_20250426_71,KAGE,NMSS,"[Biron, Musashi, Yixing, Chano, Zhang_Fei]","[Dharma, Dolia, Augran, Agudo]","[Liu_Bang, Lam, Angela, Ao_Yin, Yuhuan]","[Arli, Mai_Shiranui, Menki, Mayene]",KAGE,,,,,,,,,,
4,5,Spring_2025,Match_20250425_63,DMT,RRQ,"[Liu_Bang, Augran, Wang_Zhaojun, Consort_Yu, L...","[Mai_Shiranui, Lian_Po, Jing, Menki]","[Guan_Yu, Dian_Wei, Shi, Ao_Yin, Da_Qiao]","[Mayene, Luna, Charlotte, Biron]",DMT,,,,,,,,,,



EXPORT DATA VALIDASI MIKRO
File berhasil disimpan : ../Output/Validasi_Mikro_Sampling_5_Persen.csv


In [ ]:
# ==============================================================================
# 2D. PEMBENTUKAN DATAFRAME VALIDASI KRONOLOGIS
# HASIL RANDOM SAMPLING 5% (BAB 3.1.2.c)
# ==============================================================================

# 1. Pre-filter, urutkan Draft_Order satu kali, & format string secara vektorisasi
df_sub = (
    df_draft[df_draft['Match_ID'].isin(df_sampel_validasi['Match_ID'])]
    .sort_values('Draft_Order')
    .copy()
)

df_sub['Hero_Clean'] = df_sub['Hero_Name'].fillna('EMPTY').astype(str)
df_sub['Draft_Step'] = (
    df_sub['Draft_Order'].astype(int).map('{:02d}'.format) + '|' +
    df_sub['Team_Side'].astype(str) + '|' +
    df_sub['Is_Ban'].map({True: 'BAN', False: 'PICK'}) + '|' +
    df_sub['Hero_Clean']
)

match_groups = dict(tuple(df_sub.groupby('Match_ID', sort=False)))

# 2. Membentuk data validasi kronologis per pertandingan
data_kronologis = []
for idx, row in enumerate(df_sampel_validasi.itertuples(index=False), start=1):
    df_m = match_groups[row.Match_ID]

    teams = {
        side: df_m.loc[df_m['Team_Side'] == side, 'Team_Name'].dropna().iloc[0]
        for side in ('Blue', 'Red')
    }

    data_kronologis.append({
        'No': idx,
        'Tournament_Season': row.Tournament_Season,
        'Match_ID': row.Match_ID,
        'Blue_Team_Primer': teams['Blue'],
        'Red_Team_Primer': teams['Red'],
        # Data kronologis dari dataset primer
        'Jumlah_Aksi_Primer': len(df_m),
        'Hero_Sequence_Primer': ' -> '.join(df_m['Hero_Clean']),
        'Draft_Sequence_Primer': ' -> '.join(df_m['Draft_Step']),
        # Diisi manual berdasarkan pemeriksaan VOD & hasil validasi
        'Draft_Sequence_VOD': '',
        'Link_Sumber': '',
        'Timestamp_Awal_Draft': '',
        'Status_Validasi': '',
        'Draft_Order_Bermasalah': '',
        'Catatan': ''
    })

# 3. Menampilkan hasil DataFrame Validasi Kronologis
df_validasi_kronologis = pd.DataFrame(data_kronologis)

print("=" * 100)
print("DATAFRAME VALIDASI KRONOLOGIS - HASIL RANDOM SAMPLING 5%")
print("=" * 100)
print(f"Jumlah pertandingan sampel : {len(df_validasi_kronologis)}")
print(f"Total tindakan draft       : {df_validasi_kronologis['Jumlah_Aksi_Primer'].sum()}")
display(df_validasi_kronologis.head(5))

# 4. Export ke CSV
output_file = "../Output/Validasi_Kronologis_Sampling_5_Persen.csv"
os.makedirs(os.path.dirname(output_file), exist_ok=True)
df_validasi_kronologis.to_csv(output_file, index=False, encoding='utf-8-sig')

print("\n" + "=" * 100)
print("EXPORT DATA VALIDASI KRONOLOGIS")
print("=" * 100)
print(f"File berhasil disimpan : {output_file}")

DATAFRAME VALIDASI KRONOLOGIS - HASIL RANDOM SAMPLING 5%
Jumlah pertandingan sampel : 28
Total tindakan draft       : 504


,No,Tournament_Season,Match_ID,Blue_Team_Primer,Red_Team_Primer,Jumlah_Aksi_Primer,Hero_Sequence_Primer,Draft_Sequence_Primer,Draft_Sequence_VOD,Link_Sumber,Timestamp_Awal_Draft,Status_Validasi,Draft_Order_Bermasalah,Catatan
0,1,Spring_2025,Match_20250530_164,RRQ,DMT,18,Mayene -> Mai_Shiranui -> Liang -> Arli -> Dha...,01|Blue|BAN|Mayene -> 02|Red|BAN|Mai_Shiranui ...,,,,,,
1,2,Spring_2025,Match_20250418_29,NMSS,DMT,18,Dolia -> Mai_Shiranui -> Dharma -> Arli -> Dya...,01|Blue|BAN|Dolia -> 02|Red|BAN|Mai_Shiranui -...,,,,,,
2,3,Spring_2025,Match_20250411_7,TLN,BTR,18,Dyadia -> Arli -> Dharma -> Mai_Shiranui -> Ao...,01|Blue|BAN|Dyadia -> 02|Red|BAN|Arli -> 03|Bl...,,,,,,
3,4,Spring_2025,Match_20250426_71,KAGE,NMSS,18,Dharma -> Arli -> Dolia -> Mai_Shiranui -> Zha...,01|Blue|BAN|Dharma -> 02|Red|BAN|Arli -> 03|Bl...,,,,,,
4,5,Spring_2025,Match_20250425_63,DMT,RRQ,18,Mai_Shiranui -> Mayene -> Lian_Po -> Luna -> L...,01|Blue|BAN|Mai_Shiranui -> 02|Red|BAN|Mayene ...,,,,,,



EXPORT DATA VALIDASI KRONOLOGIS
File berhasil disimpan : ../Output/Validasi_Kronologis_Sampling_5_Persen.csv


In [ ]:
# ==============================================================================
# 2E. IDENTIFIKASI ANOMALI, PEMBERSIHAN DATA & STATISTIK DESKRIPTIF (BAB 3.1.2.d)
# ==============================================================================
print("="*60)
print("IDENTIFIKASI ANOMALI & PEMBERSIHAN DATA (BAB 3.1.2.d)")
print("="*60)

initial_matches = df_draft['Match_ID'].nunique()
log_anomali = []

# Fungsi bantu untuk mencatat detail Match_ID yang terindikasi anomali
def catat_anomali(df_sumber, list_match_id, alasan):
    for m_id in list_match_id:
        sub = df_sumber[df_sumber['Match_ID'] == m_id]
        musim = sub['Tournament_Season'].iloc[0] if not sub.empty else '-'
        tanggal = sub['Tanggal_Match'].iloc[0] if not sub.empty else '-'
        tim = " vs ".join(sub['Team_Name'].dropna().unique())
        log_anomali.append({
            'Tournament_Season': musim,
            'Tanggal_Match': tanggal,
            'Match_ID': m_id,
            'Pertandingan': tim,
            'Jumlah_Baris_Awal': len(sub),
            'Hero_Unik_Tercatat': sub['Hero_ID'].nunique(dropna=True),
            'Alasan_Eksklusi': alasan
        })

# TAHAP 1: Deteksi Missing Value (No Ban Penalty & Ultimate Battle / Blind Pick)
matches_missing = df_draft[df_draft['Hero_ID'].isnull() | df_draft['Hero_Name'].isnull()]['Match_ID'].unique()
catat_anomali(df_draft, matches_missing, "Missing Value (Ultimate Battle / No Ban Penalty)")
df_draft = df_draft[~df_draft['Match_ID'].isin(matches_missing)].copy()
print(f"1. Dihapus karena Missing Value (Ultimate Battle/Penalty) : {len(matches_missing)} pertandingan")
if len(matches_missing) > 0:
    print(f"   -> Daftar Match_ID: {list(matches_missing)}")

# TAHAP 2: Deteksi Jumlah Aksi Tidak Lengkap (Harus tepat 18 aksi per pertandingan)
match_action_counts = df_draft.groupby('Match_ID').size()
invalid_action_matches = match_action_counts[match_action_counts != 18].index
catat_anomali(df_draft, invalid_action_matches, "Jumlah Aksi Tidak Lengkap (!= 18 Aksi)")
df_draft = df_draft[~df_draft['Match_ID'].isin(invalid_action_matches)].copy()
print(f"2. Dihapus karena jumlah aksi != 18                       : {len(invalid_action_matches)} pertandingan")
if len(invalid_action_matches) > 0:
    print(f"   -> Daftar Match_ID: {list(invalid_action_matches)}")

# TAHAP 3: Deteksi Hero Duplikat / Mirror Pick dalam 1 Pertandingan (Harus 18 Hero Unik)
hero_unique_counts = df_draft.groupby('Match_ID')['Hero_ID'].nunique()
invalid_unique_matches = hero_unique_counts[hero_unique_counts != 18].index
catat_anomali(df_draft, invalid_unique_matches, "Hero Duplikat / Mirror Pick (< 18 Hero Unik)")
df_draft = df_draft[~df_draft['Match_ID'].isin(invalid_unique_matches)].copy()
print(f"3. Dihapus karena Hero Duplikat / Mirror Pick             : {len(invalid_unique_matches)} pertandingan")
if len(invalid_unique_matches) > 0:
    print(f"   -> Daftar Match_ID: {list(invalid_unique_matches)}")

# Menampilkan Tabel Log Anomali jika ada pertandingan yang dihapus
print("\n" + "="*60)
print("RINCIAN PERTANDINGAN YANG DIEKSKLUSI (LOG ANOMALI)")
print("="*60)
df_log_anomali = pd.DataFrame(log_anomali)
if not df_log_anomali.empty:
    display(df_log_anomali)
else:
    print("Tidak ada pertandingan yang dieksklusi (Seluruh data sudah bersih).")

# Memastikan tipe data Hero_ID kembali menjadi integer setelah NaN dihapus
df_draft['Hero_ID'] = df_draft['Hero_ID'].astype(int)

# Mengurutkan dan men-generate ulang Match_Sequence & NO secara kontinu pada Clean Dataset
df_draft['Match_Sequence'] = pd.factorize(df_draft['Match_ID'])[0] + 1
df_draft = df_draft.sort_values(by=['Match_Sequence', 'Draft_Order']).reset_index(drop=True)
df_draft['NO'] = range(1, len(df_draft) + 1)

final_matches = df_draft['Match_ID'].nunique()
print("-" * 60)
print(f"Total Pertandingan Awal                   : {initial_matches}")
print(f"Total Pertandingan Dihapus (Anomali)      : {initial_matches - final_matches}")
print(f"Total Pertandingan Valid (Clean Dataset)  : {final_matches} ({len(df_draft)} baris)")

# Menampilkan Tabel Statistik Deskriptif per Musim (Sesuai Tabel 3.2 BAB 3)
print("\n" + "="*60)
print("STATISTIK DESKRIPTIF PER MUSIM IKL (CLEAN DATASET)")
print("="*60)

rekap_musim = []
for musim in ['Spring_2025', 'Fall_2025', 'Spring_2026']:
    df_m = df_draft[df_draft['Tournament_Season'] == musim]
    rekap_musim.append({
        'Tournament_Season': musim,
        'Jumlah_Pertandingan': df_m['Match_ID'].nunique(),
        'Jumlah_Baris_Draft': len(df_m),
        'Hero_Unik_Muncul': df_m['Hero_ID'].nunique()
    })

df_rekap_musim = pd.DataFrame(rekap_musim)
display(df_rekap_musim)

IDENTIFIKASI ANOMALI & PEMBERSIHAN DATA (BAB 3.1.2.d)
1. Dihapus karena Missing Value (Ultimate Battle/Penalty) : 10 pertandingan
   -> Daftar Match_ID: ['Match_20250517_140', 'Match_20250601_187', 'Match_20250906_56', 'Match_20250913_81', 'Match_20251005_177', 'Match_20260306_37', 'Match_20260308_50', 'Match_20260315_86', 'Match_20260405_166', 'Match_20260411_196']
2. Dihapus karena jumlah aksi != 18                       : 0 pertandingan
3. Dihapus karena Hero Duplikat / Mirror Pick             : 1 pertandingan
   -> Daftar Match_ID: ['Match_20250413_23']

RINCIAN PERTANDINGAN YANG DIEKSKLUSI (LOG ANOMALI)


,Tournament_Season,Tanggal_Match,Match_ID,Pertandingan,Jumlah_Baris_Awal,Hero_Unik_Tercatat,Alasan_Eksklusi
0,Spring_2025,5/17/2025,Match_20250517_140,NMSS vs VESA,18,14,Missing Value (Ultimate Battle / No Ban Penalty)
1,Spring_2025,6/1/2025,Match_20250601_187,DMT vs KAGE,18,7,Missing Value (Ultimate Battle / No Ban Penalty)
2,Fall_2025,9/6/2025,Match_20250906_56,TLN vs RVM,18,14,Missing Value (Ultimate Battle / No Ban Penalty)
3,Fall_2025,9/13/2025,Match_20250913_81,ONIC vs AE,18,14,Missing Value (Ultimate Battle / No Ban Penalty)
4,Fall_2025,10/5/2025,Match_20251005_177,BTR vs KAGE,18,7,Missing Value (Ultimate Battle / No Ban Penalty)
5,Spring_2026,3/6/2026,Match_20260306_37,BTR vs ONIC,18,14,Missing Value (Ultimate Battle / No Ban Penalty)
6,Spring_2026,3/8/2026,Match_20260308_50,KAGE vs SE,18,17,Missing Value (Ultimate Battle / No Ban Penalty)
7,Spring_2026,3/15/2026,Match_20260315_86,AE vs RRQ,18,17,Missing Value (Ultimate Battle / No Ban Penalty)
8,Spring_2026,4/5/2026,Match_20260405_166,RRQ vs DMT,18,8,Missing Value (Ultimate Battle / No Ban Penalty)
9,Spring_2026,4/11/2026,Match_20260411_196,OG vs KAGE,18,9,Missing Value (Ultimate Battle / No Ban Penalty)


------------------------------------------------------------
Total Pertandingan Awal                   : 569
Total Pertandingan Dihapus (Anomali)      : 11
Total Pertandingan Valid (Clean Dataset)  : 558 (10044 baris)

STATISTIK DESKRIPTIF PER MUSIM IKL (CLEAN DATASET)


,Tournament_Season,Jumlah_Pertandingan,Jumlah_Baris_Draft,Hero_Unik_Muncul
0,Spring_2025,184,3312,79
1,Fall_2025,174,3132,74
2,Spring_2026,200,3600,82


In [ ]:
# ==============================================================================
# 4. DATA INTEGRATION & PENENTUAN STATUS KEMENANGAN (BAB 3.1.3.a)
# ==============================================================================
# 1. Integrasi referensi atribut hero dari Hero_List
df_draft = pd.merge(
    df_draft, 
    df_hero[['Hero_ID', 'Hero_Name', 'Hero_Tipe', 'Default_Role']], 
    on='Hero_ID', 
    how='left', 
    suffixes=('', '_ref')
)

# Menjamin konsistensi nama dan tipe hero dengan kamus Hero_List
df_draft['Hero_Name'] = df_draft['Hero_Name_ref'].fillna(df_draft['Hero_Name'])
df_draft['Hero_Tipe'] = df_draft['Hero_Tipe_ref'].fillna(df_draft['Hero_Tipe'])
if 'Hero_Role' not in df_draft.columns:
    df_draft['Hero_Role'] = df_draft['Default_Role']
else:
    df_draft['Hero_Role'] = df_draft['Hero_Role'].fillna(df_draft['Default_Role'])

# Menghapus kolom bantu hasil penggabungan
df_draft.drop(columns=['Hero_Name_ref', 'Hero_Tipe_ref', 'Default_Role'], errors='ignore', inplace=True)

# 2. Ekstraksi logika penentuan status kemenangan tim (Is_Win)
def check_win(row):
    outcome = str(row['Match_Outcome']).strip().lower()
    side = str(row['Team_Side']).strip().lower()
    if side == 'blue' and outcome == 'blue_win':
        return True
    elif side == 'red' and outcome == 'red_win':
        return True
    return False

df_draft['Is_Win'] = df_draft.apply(check_win, axis=1)

print("="*60)
print("DATA INTEGRATION SELESAI")
print("="*60)
print(f"Jumlah Match Bersih   : {df_draft['Match_ID'].nunique()}")
print(f"Jumlah Hero Referensi : {df_hero['Hero_ID'].nunique()}")
print(f"Jumlah Draft Action   : {len(df_draft)}")
print(f"Draft per Match       : {len(df_draft) // df_draft['Match_ID'].nunique()}")

DATA INTEGRATION SELESAI
Jumlah Match Bersih   : 558
Jumlah Hero Referensi : 113
Jumlah Draft Action   : 10044
Draft per Match       : 18


In [ ]:
# Membuat tampilan ringkasan hasil integrasi 3 musim seperti Gambar 3.4 pada BAB 3
df_tampil = df_draft.rename(columns={'NO': 'No'})
kolom_integrasi = [
    'No', 'Tanggal_Match', 'Match_ID', 'Match_Sequence', 'Tournament_Season',
    'Team_Name', 'Team_Side', 'Draft_Order', 'Is_Ban', 'Hero_ID',
    'Hero_Name', 'Hero_Tipe', 'Hero_Role', 'Match_Outcome', 'Is_Win'
]
df_tampil = df_tampil[kolom_integrasi]

s_25 = df_tampil[df_tampil['Tournament_Season'] == 'Spring_2025'].head(3)
f_25 = df_tampil[df_tampil['Tournament_Season'] == 'Fall_2025'].head(3)
s_26 = df_tampil[df_tampil['Tournament_Season'] == 'Spring_2026'].head(3)
baris_titik = pd.DataFrame([{col: '...' for col in kolom_integrasi}])

tabel_integrasi_visual = pd.concat([s_25, baris_titik, f_25, baris_titik, s_26], ignore_index=True)

print("="*60)
print(" HASIL INTEGRASI DATA 3 MUSIM IKL")
print("="*60)
display(tabel_integrasi_visual)

 HASIL INTEGRASI DATA 3 MUSIM IKL


,No,Tanggal_Match,Match_ID,Match_Sequence,Tournament_Season,Team_Name,Team_Side,Draft_Order,Is_Ban,Hero_ID,Hero_Name,Hero_Tipe,Hero_Role,Match_Outcome,Is_Win
0,1,4/11/2025,Match_20250411_1,1,Spring_2025,RRQ,Blue,1,True,564,Mayene,Fighter,Clash_Lane,Red_Win,False
1,2,4/11/2025,Match_20250411_1,1,Spring_2025,NMSS,Red,2,True,159,Dolia,Support,Roamer,Red_Win,True
2,3,4/11/2025,Match_20250411_1,1,Spring_2025,RRQ,Blue,3,True,108,Mozi,Mage,Mid_Lane,Red_Win,False
3,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,3313,8/22/2025,Match_20250822_1,185,Fall_2025,AE,Blue,1,True,176,Yuhuan,Mage,Mid_Lane,Red_Win,False
5,3314,8/22/2025,Match_20250822_1,185,Fall_2025,VESA,Red,2,True,146,Luna,Fighter,Jungler,Red_Win,True
6,3315,8/22/2025,Match_20250822_1,185,Fall_2025,AE,Blue,3,True,134,Dharma,Fighter,Clash_Lane,Red_Win,False
7,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8,6445,2/27/2026,Match_20260227_1,359,Spring_2026,ONIC,Blue,1,True,172,Chicha,Fighter,Clash_Lane,Blue_Win,True
9,6446,2/27/2026,Match_20260227_1,359,Spring_2026,SE,Red,2,True,140,Guan_Yu,Fighter,Clash_Lane,Blue_Win,False


In [ ]:
# ==============================================================================
# 5. ENCODING & PEMBENTUKAN HERO VOCABULARY (BAB 3.1.3.b)
# ==============================================================================
encoder = LabelEncoder()
encoder.fit(df_hero['Hero_ID'])

# Menambahkan +1 karena indeks 0 didedikasikan untuk nilai kosong (padding) pada LSTM
TOTAL_HEROES_VOCAB = len(encoder.classes_) + 1 
print(f"Total Hero dalam Vocabulary (termasuk padding 0): {TOTAL_HEROES_VOCAB}")

# Menerapkan transformasi ke dataset Hero dan Draft Pick
df_hero['Encoded_Hero'] = encoder.transform(df_hero['Hero_ID']) + 1
df_draft['Encoded_Hero'] = encoder.transform(df_draft['Hero_ID']) + 1

display(df_hero[['Hero_ID', 'Hero_Name', 'Encoded_Hero']].sort_values(by='Hero_Name').head(10))
print("\nHasil encoding pada dataset draft:")
display(df_draft[['Match_Sequence', 'Draft_Order', 'Hero_ID', 'Hero_Name', 'Encoded_Hero']])

Total Hero dalam Vocabulary (termasuk padding 0): 114


,Hero_ID,Hero_Name,Encoded_Hero
0,533,Agudo,97
1,545,Alessio,103
2,514,Allain,88
3,142,Angela,35
4,519,Ao_Yin,90
5,116,Arke,12
6,199,Arli,77
7,166,Arthur,49
8,556,Ata,105
9,183,Athena,65



Hasil encoding pada dataset draft:


,Match_Sequence,Draft_Order,Hero_ID,Hero_Name,Encoded_Hero
0,1,1,564,Mayene,108
1,1,2,159,Dolia,46
2,1,3,108,Mozi,4
3,1,4,134,Dharma,29
4,1,5,157,Mai_Shiranui,45
...,...,...,...,...,...
10039,558,14,111,Lady_Sun,7
10040,558,15,536,Charlotte,99
10041,558,16,110,Garuda,6
10042,558,17,128,Fatih,23


In [ ]:
# ==============================================================================
# 6A. TRANSFORMASI MATRIKS KHUSUS (ARM)
# ==============================================================================
df_picks = df_draft[df_draft['Is_Ban'] == False].copy()
df_bans  = df_draft[df_draft['Is_Ban'] == True].copy()

print(f"Data ARM disiapkan: {len(df_picks)} Picks, {len(df_bans)} Bans")

Data ARM disiapkan: 5580 Picks, 4464 Bans


In [ ]:
# ==============================================================================
# 6B. TRANSFORMASI MATRIKS KHUSUS (MLP - MULTI-HOT ENCODING)
# ==============================================================================
blue_picks = df_picks[df_picks['Team_Side'] == 'Blue'].groupby('Match_ID', sort=False)['Encoded_Hero'].apply(list)
red_picks  = df_picks[df_picks['Team_Side'] == 'Red'].groupby('Match_ID', sort=False)['Encoded_Hero'].apply(list)
outcomes   = df_picks.groupby('Match_ID', sort=False)['Match_Outcome'].first()

print("Contoh data Blue Team:")
display(blue_picks.head(5))
print("\nContoh data Red Team:")
display(red_picks.head(5))

def encode_match_optimized(blue_heroes, red_heroes, total_dim):
    feature_vector = np.zeros(total_dim)
    if isinstance(blue_heroes, list):
        for hero_idx in blue_heroes:
            feature_vector[hero_idx] = 1
    if isinstance(red_heroes, list):
        for hero_idx in red_heroes:
            feature_vector[hero_idx] = -1
    return feature_vector

X_mlp, y_mlp = [], []

for match_id in outcomes.index:
    blue = blue_picks.get(match_id, [])
    red  = red_picks.get(match_id, [])
    
    X_mlp.append(encode_match_optimized(blue, red, TOTAL_HEROES_VOCAB))
    match_result = 1 if outcomes[match_id] == 'Blue_Win' else 0
    y_mlp.append(match_result)

X_mlp = np.array(X_mlp)
y_mlp = np.array(y_mlp)

print(f"\nBentuk Array Input MLP (X_mlp)  : {X_mlp.shape}")
print(f"Bentuk Array Target MLP (y_mlp) : {y_mlp.shape}")

Contoh data Blue Team:


Match_ID
Match_20250411_1      [45, 7, 78, 86, 76]
Match_20250411_2     [44, 90, 40, 25, 86]
Match_20250411_3    [46, 28, 108, 72, 22]
Match_20250411_4     [77, 76, 64, 10, 92]
Match_20250411_5     [89, 60, 54, 35, 36]
Name: Encoded_Hero, dtype: object


Contoh data Red Team:


Match_ID
Match_20250411_1    [77, 109, 38, 64, 95]
Match_20250411_2     [54, 80, 75, 96, 60]
Match_20250411_3      [29, 1, 90, 89, 40]
Match_20250411_4     [98, 29, 60, 75, 24]
Match_20250411_5    [90, 21, 22, 70, 108]
Name: Encoded_Hero, dtype: object


Bentuk Array Input MLP (X_mlp)  : (558, 114)
Bentuk Array Target MLP (y_mlp) : (558,)


In [ ]:
# ==============================================================================
# 6C. TRANSFORMASI MATRIKS KHUSUS (LSTM - SEQUENCE PADDING & GROUPS)
# ==============================================================================
sequences = []
targets   = []
groups    = []

for match_seq, match_data in df_draft.groupby('Match_Sequence'):
    hero_sequence = match_data['Encoded_Hero'].tolist()
    
    for i in range(1, len(hero_sequence)):
        seq    = hero_sequence[:i]     
        target = hero_sequence[i]   
        sequences.append(seq)
        targets.append(target)
        groups.append(match_seq) # Penanda grup pertandingan untuk GroupKFold

MAX_DRAFT_LENGTH = df_draft.groupby('Match_Sequence').size().max()
MAX_INPUT_LENGTH = MAX_DRAFT_LENGTH - 1 # 17 langkah

X_lstm      = pad_sequences(sequences, maxlen=MAX_INPUT_LENGTH, padding='pre')
y_lstm      = np.array(targets)
groups_lstm = np.array(groups)

print(f"Bentuk Input LSTM (X_lstm)        : {X_lstm.shape}")
print(f"Bentuk Target LSTM (y_lstm)       : {y_lstm.shape}")
print(f"Bentuk Groups LSTM (groups_lstm)  : {groups_lstm.shape}")

Bentuk Input LSTM (X_lstm)        : (9486, 17)
Bentuk Target LSTM (y_lstm)       : (9486,)
Bentuk Groups LSTM (groups_lstm)  : (9486,)


In [ ]:
# ==============================================================================
# 7. PERHITUNGAN STATISTIK & AGREGAT HERO (PER MUSIM MENTAH & DATASET UTAMA BERSIH)
# ==============================================================================
kolom_dasar_hero = ['NO', 'Hero_ID', 'Hero_Name', 'Hero_Tipe', 'Default_Role', 'Encoded_Hero']
df_hero_base = df_hero[kolom_dasar_hero].copy()

def hitung_statistik_hero(df_sumber, df_ref_base, judul_statistik):
    print("=" * 70)
    print(f"STATISTIK DAN AGREGAT: {judul_statistik}")
    print("=" * 70)
    
    # 1. Salin data dan abaikan baris kosong (NaN) pada Hero_ID
    df_valid = df_sumber.dropna(subset=['Hero_ID']).copy()
    df_valid['Hero_ID'] = df_valid['Hero_ID'].astype(int)
    
    # 2. Hitung atribut Is_Win secara otomatis jika belum tersedia
    if 'Is_Win' not in df_valid.columns:
        side = df_valid['Team_Side'].astype(str).str.strip().str.lower()
        outcome = df_valid['Match_Outcome'].astype(str).str.strip().str.lower()
        df_valid['Is_Win'] = ((side == 'blue') & (outcome == 'blue_win')) | ((side == 'red') & (outcome == 'red_win'))
    
    # 3. Pisahkan aksi Pick dan Ban
    df_picks_sub = df_valid[df_valid['Is_Ban'] == False]
    df_bans_sub  = df_valid[df_valid['Is_Ban'] == True]
    
    # 4. Hitung Total Picks & Total Wins
    stats_pick = df_picks_sub.groupby('Hero_ID').agg(
        Total_Picks=('Hero_ID', 'count'),
        Total_Wins=('Is_Win', 'sum')
    ).reset_index()
    
    # 5. Hitung Total Bans
    stats_ban = df_bans_sub.groupby('Hero_ID').agg(
        Total_Bans=('Hero_ID', 'count')
    ).reset_index()
    
    
    
    # 6. Hitung Total Loses dan Win Rate
    stats_pick['Total_Loses'] = stats_pick['Total_Picks'] - stats_pick['Total_Wins']
    stats_pick['Win_Rate']    = stats_pick['Total_Wins'] / stats_pick['Total_Picks']
    
    # 7. Gabungkan hasil perhitungan ke tabel referensi hero
    df_hasil = pd.merge(df_ref_base, stats_pick, on='Hero_ID', how='left')
    df_hasil = pd.merge(df_hasil, stats_ban, on='Hero_ID', how='left')
    
    # 8. Isi hero yang belum pernah di-pick/ban dengan nilai 0
    df_hasil.fillna({
        'Total_Picks': 0, 
        'Total_Bans': 0, 
        'Total_Wins': 0, 
        'Total_Loses': 0, 
        'Win_Rate': 0.0
    }, inplace=True)
    
    # 9. Pastikan tipe data jumlah aksi berupa integer
    for col in ['Total_Picks', 'Total_Bans', 'Total_Wins', 'Total_Loses']:
        df_hasil[col] = df_hasil[col].astype(int)
    
    # 10. Total gabungan aksi (Picks + Bans) untuk setiap hero
    df_hasil['Total_Bans_Picks'] = df_hasil['Total_Picks'] + df_hasil['Total_Bans']
        
    # 11. Urutkan kolom dan baris
    kolom_urut = [
        'NO', 'Hero_ID', 'Hero_Name', 'Hero_Tipe', 'Default_Role', 
        'Encoded_Hero', 'Total_Picks', 'Total_Bans', 'Total_Wins', 'Total_Loses', 'Total_Bans_Picks','Win_Rate'
    ]
    df_hasil = df_hasil[kolom_urut].sort_values(
        by=['Total_Picks', 'Win_Rate'], 
        ascending=[False, False]
    ).reset_index(drop=True)
    
    # --- METRIK TAMBAHAN ---
    hero_muncul = df_hasil[(df_hasil['Total_Picks'] > 0) | (df_hasil['Total_Bans'] > 0)]
    total_hero_unik = len(hero_muncul)
    
    print(f"Total Pertandingan Terhitung : {df_sumber['Match_ID'].nunique()} Match")
    print(f"Total Picks Terhitung        : {df_hasil['Total_Picks'].sum()} Picks")
    print(f"Total Bans Terhitung         : {df_hasil['Total_Bans'].sum()} Bans")
    print(f"Total Hero Unik Muncul       : {total_hero_unik} Hero")
    print("-" * 70)
    
    display(df_hasil.head(10))
    return df_hasil

# Eksekusi fungsi untuk masing-masing musim
df_hero_spring_25 = hitung_statistik_hero(df_spring_25, df_hero_base, "IKL Spring 2025")
df_hero_fall_25 = hitung_statistik_hero(df_fall_25, df_hero_base, "IKL Fall 2025")
df_hero_spring_26 = hitung_statistik_hero(df_spring_26, df_hero_base, "IKL Spring 2026")
df_hero = hitung_statistik_hero(df_draft, df_hero_base, "DATASET UTAMA - Clean Dataset (Gabungan Ketiga Musim)")

STATISTIK DAN AGREGAT: IKL Spring 2025
Total Pertandingan Terhitung : 187 Match
Total Picks Terhitung        : 1870 Picks
Total Bans Terhitung         : 1484 Bans
Total Hero Unik Muncul       : 79 Hero
----------------------------------------------------------------------


,NO,Hero_ID,Hero_Name,Hero_Tipe,Default_Role,Encoded_Hero,Total_Picks,Total_Bans,Total_Wins,Total_Loses,Total_Bans_Picks,Win_Rate
0,5,519,Ao_Yin,Marksman,Farm_Lane,90,92,38,41,51,130,0.445652
1,109,171,Zhang_Fei,Support,Roamer,54,79,69,45,34,148,0.569620
2,7,199,Arli,Marksman,Farm_Lane,77,72,100,34,38,172,0.472222
3,74,564,Mayene,Fighter,Clash_Lane,108,71,49,36,35,120,0.507042
4,106,197,Yixing,Mage,Mid_Lane,75,70,55,32,38,125,0.457143
5,76,198,Menki,Fighter,Jungler,76,67,42,34,33,109,0.507463
6,16,177,Chano,Marksman,Farm_Lane,60,65,22,35,30,87,0.538462
7,13,503,Biron,Fighter,Clash_Lane,80,65,25,30,35,90,0.461538
8,63,105,Lian_Po,Tank,Clash_Lane,1,64,30,25,39,94,0.390625
9,24,134,Dharma,Fighter,Clash_Lane,29,63,110,35,28,173,0.555556


STATISTIK DAN AGREGAT: IKL Fall 2025
Total Pertandingan Terhitung : 177 Match
Total Picks Terhitung        : 1770 Picks
Total Bans Terhitung         : 1400 Bans
Total Hero Unik Muncul       : 74 Hero
----------------------------------------------------------------------


,NO,Hero_ID,Hero_Name,Hero_Tipe,Default_Role,Encoded_Hero,Total_Picks,Total_Bans,Total_Wins,Total_Loses,Total_Bans_Picks,Win_Rate
0,108,176,Yuhuan,Mage,Mid_Lane,59,91,57,54,37,148,0.593407
1,24,134,Dharma,Fighter,Clash_Lane,29,74,69,41,33,143,0.554054
2,5,519,Ao_Yin,Marksman,Farm_Lane,90,74,40,40,34,114,0.540541
3,7,199,Arli,Marksman,Farm_Lane,77,68,88,32,36,156,0.470588
4,86,502,Pei,Assasin,Jungler,79,67,76,35,32,143,0.522388
5,109,171,Zhang_Fei,Support,Roamer,54,66,40,37,29,106,0.560606
6,25,133,Di_Renjie,Marksman,Farm_Lane,28,61,36,33,28,97,0.540984
7,74,564,Mayene,Fighter,Clash_Lane,108,58,49,30,28,107,0.517241
8,16,177,Chano,Marksman,Farm_Lane,60,56,26,29,27,82,0.517857
9,58,127,Lady_Zhen,Mage,Mid_Lane,22,55,32,32,23,87,0.581818


STATISTIK DAN AGREGAT: IKL Spring 2026
Total Pertandingan Terhitung : 205 Match
Total Picks Terhitung        : 2050 Picks
Total Bans Terhitung         : 1618 Bans
Total Hero Unik Muncul       : 82 Hero
----------------------------------------------------------------------


,NO,Hero_ID,Hero_Name,Hero_Tipe,Default_Role,Encoded_Hero,Total_Picks,Total_Bans,Total_Wins,Total_Loses,Total_Bans_Picks,Win_Rate
0,60,168,Lapulapu,Support,Roamer,51,89,30,50,39,119,0.561798
1,109,171,Zhang_Fei,Support,Roamer,54,81,51,41,40,132,0.506173
2,5,519,Ao_Yin,Marksman,Farm_Lane,90,74,85,36,38,159,0.486486
3,16,177,Chano,Marksman,Farm_Lane,60,73,18,31,42,91,0.424658
4,13,503,Biron,Fighter,Clash_Lane,80,72,27,34,38,99,0.472222
5,24,134,Dharma,Fighter,Clash_Lane,29,71,70,37,34,141,0.521127
6,72,157,Mai_Shiranui,Assasin,Mid_Lane,45,62,78,33,29,140,0.532258
7,18,172,Chicha,Fighter,Clash_Lane,55,56,64,31,25,120,0.553571
8,86,502,Pei,Assasin,Jungler,79,51,112,32,19,163,0.627451
9,38,584,Flowborn_Marksman,Marksman,Farm_Lane,112,49,14,34,15,63,0.693878


STATISTIK DAN AGREGAT: DATASET UTAMA - Clean Dataset (Gabungan Ketiga Musim)
Total Pertandingan Terhitung : 558 Match
Total Picks Terhitung        : 5580 Picks
Total Bans Terhitung         : 4464 Bans
Total Hero Unik Muncul       : 100 Hero
----------------------------------------------------------------------


,NO,Hero_ID,Hero_Name,Hero_Tipe,Default_Role,Encoded_Hero,Total_Picks,Total_Bans,Total_Wins,Total_Loses,Total_Bans_Picks,Win_Rate
0,5,519,Ao_Yin,Marksman,Farm_Lane,90,233,162,113,120,395,0.484979
1,109,171,Zhang_Fei,Support,Roamer,54,222,160,121,101,382,0.545045
2,24,134,Dharma,Fighter,Clash_Lane,29,199,245,108,91,444,0.542714
3,16,177,Chano,Marksman,Farm_Lane,60,189,65,93,96,254,0.492063
4,13,503,Biron,Fighter,Clash_Lane,80,186,67,90,96,253,0.483871
5,108,176,Yuhuan,Mage,Mid_Lane,59,183,89,106,77,272,0.579235
6,7,199,Arli,Marksman,Farm_Lane,77,169,223,84,85,392,0.497041
7,63,105,Lian_Po,Tank,Clash_Lane,1,161,64,70,91,225,0.434783
8,106,197,Yixing,Mage,Mid_Lane,75,160,114,75,85,274,0.468750
9,86,502,Pei,Assasin,Jungler,79,159,244,87,72,403,0.547170


In [ ]:
# ==============================================================================
# 8. PENYIMPANAN DATASET TERSTANDARISASI
# ==============================================================================
# Menyimpan array NumPy untuk model MLP dan LSTM
np.save('../Data/X_mlp.npy', X_mlp)
np.save('../Data/y_mlp.npy', y_mlp)
np.save('../Data/X_lstm.npy', X_lstm)
np.save('../Data/y_lstm.npy', y_lstm)
np.save('../Data/groups_lstm.npy', groups_lstm)

# Menyimpan DataFrames bersih untuk ARM dan Dataset Utama
df_draft.to_csv('../Data/draft_combined_cleaned.csv', index=False)
df_picks.to_csv('../Data/arm_picks_cleaned.csv', index=False)
df_bans.to_csv('../Data/arm_bans_cleaned.csv', index=False)

# Menyimpan data statistik hero (Dataset Utama & masing-masing musim)
df_hero.to_csv('../Data/hero_data.csv', index=False)
df_hero_spring_25.to_csv('../Data/hero_stats_Spring_2025.csv', index=False)
df_hero_fall_25.to_csv('../Data/hero_stats_Fall_2025.csv', index=False)
df_hero_spring_26.to_csv('../Data/hero_stats_Spring_2026.csv', index=False)

# Menyimpan objek Encoder & parameter global
with open('../Data/global_parameters.pkl', 'wb') as f:
    pickle.dump({
        'encoder': encoder, 
        'TOTAL_HEROES_VOCAB': TOTAL_HEROES_VOCAB, 
        'MAX_INPUT_LENGTH': MAX_INPUT_LENGTH
    }, f)

print("Seluruh dataset dan parameter berhasil disimpan ke folder Data!")

Seluruh dataset dan parameter berhasil disimpan ke folder Data!


In [ ]:
print("Contoh Input LSTM (18 baris pertama):")
print(X_lstm[:18])
print("="*60)
print("Contoh Target LSTM (18 baris pertama):")
print("="*60)
print(y_lstm[:18])
print("="*60)
print("Contoh Groups LSTM (18 baris pertama):")
print("="*60)
print(groups_lstm[:18])
print("="*60)
print("Distribusi jumlah sampel sekuens per Match_Sequence:")
display(pd.Series(groups_lstm).value_counts().sort_index())

Contoh Input LSTM (18 baris pertama):
[[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 108]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 108  46]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0 108  46   4]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0 108  46   4  29]
 [  0   0   0   0   0   0   0   0   0   0   0   0 108  46   4  29  45]
 [  0   0   0   0   0   0   0   0   0   0   0 108  46   4  29  45  77]
 [  0   0   0   0   0   0   0   0   0   0 108  46   4  29  45  77 109]
 [  0   0   0   0   0   0   0   0   0 108  46   4  29  45  77 109   7]
 [  0   0   0   0   0   0   0   0 108  46   4  29  45  77 109   7  78]
 [  0   0   0   0   0   0   0 108  46   4  29  45  77 109   7  78  38]
 [  0   0   0   0   0   0 108  46   4  29  45  77 109   7  78  38  80]
 [  0   0   0   0   0 108  46   4  29  45  77 109   7  78  38  80  25]
 [  0   0   0   0 108  46   4  29  45  77 109   7  78  38  80  25  89]
 [  0   0   0 108  46   4  29  45  77 1

1      17
2      17
3      17
4      17
5      17
       ..
554    17
555    17
556    17
557    17
558    17
Name: count, Length: 558, dtype: int64